In [1]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import pyarrow
import config

from src.data_loading import load_crsp, load_futures, load_crsp_polars, wrds_fetch
from src.preprocessing import clean_crsp, clean_futures
from src.feature_engineering import add_target, add_volatility_momentum, crosssectional_rank, get_feature_cols 


plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)
print('Config window:', config.START_DATE, '→', config.END_DATE)

Config window: 2000-01-01 → 2020-11-30


In [2]:
data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(data.dtypes)
data.head()

(19603583, 4)
['PERMNO', 'date', 'ret', 'mkt_ret']
PERMNO              int32
date       datetime64[ms]
ret               float64
mkt_ret           float64
dtype: object


,PERMNO,date,ret,mkt_ret
0,10001,2000-01-03,0.0074,-0.0095
1,10001,2000-01-04,-0.0146,-0.0383
2,10001,2000-01-05,0.0148,0.0019
3,10001,2000-01-06,-0.0073,0.0010
4,10001,2000-01-07,-0.0074,0.0271


---
Features Engineering

| # | Feature | Description |
|---|------|--------|
| 1 | Volatility (Windows = [21, 63, 252] )|  |
| 2 | Momentum ([5, 21, 63, 126, 252] ) |  |
| 3 | Previous Days Returns ([1, 5, 25]) |  |

---
# Momentum, Volatility, Target

In [4]:
from src.feature_engineering import build_features

df = build_features(data, config.VALUE_RETURN, trading_interval=False)
 
df.head()

c:\Users\dario\Documents\Academique\EPFL\Master\MA-2\Machine Learning for Finance\ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\dario\Documents\Academique\EPFL\Master\MA-2\Machine Learning for Finance\ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,PERMNO,date,ret,mkt_ret,reversal_1d,mom_scaled_5d,mom_5d,vol_5d,mom_scaled_21d,mom_21d,...,mom_scaled_63d,mom_63d,vol_63d,mom_scaled_126d,mom_126d,vol_126d,mom_scaled_252d,mom_252d,vol_252d,target
201,10001,2000-10-18,-0.0071,-0.0058,0.0478,-0.1288,-0.0149,-0.2534,0.2205,0.2225,...,0.3624,0.3400,-0.2376,0.2460,0.2415,-0.2669,0.1693,0.1433,-0.3210,0.0000
202,10001,2000-10-19,0.0000,0.0347,-0.0238,0.2008,0.1983,-0.4209,0.3127,0.3083,...,0.3661,0.3450,-0.2372,0.2283,0.2256,-0.2663,0.1663,0.1418,-0.3216,0.0071
203,10001,2000-10-20,0.0071,0.0059,-0.1657,-0.0070,-0.0096,-0.4258,0.2253,0.2244,...,0.3036,0.2897,-0.2471,0.1929,0.1921,-0.2678,0.1536,0.1306,-0.3226,-0.0071
204,10001,2000-10-23,-0.0071,-0.0008,0.0797,0.1743,0.0773,-0.4107,0.2316,0.2256,...,0.3008,0.2812,-0.2474,0.1935,0.1894,-0.2682,0.1575,0.1342,-0.3244,0.0143
205,10001,2000-10-24,0.0143,0.0017,-0.1373,-0.3645,-0.1235,-0.4532,0.1132,0.1310,...,0.3197,0.2889,-0.2476,0.2450,0.2379,-0.2690,0.1488,0.1236,-0.3256,-0.0141


In [5]:
df.to_parquet(config.FEATURES_PATH_CLEAN)

---
#  Batch

In [ ]:
features = pd.read_parquet(config.FEATURES_PATH_CLEAN)
print(f'Number of unique PERMNO in dataset : {features['PERMNO'].nunique()}')
from src.utils import generate_batches, train_val_test_split, split_batch

split_batch_dict = split_batch(features, 'PERMNO', 'date', config.BATCH_NUMBER, config.BATCH_SIZE, overlap=False)

Number of unique PERMNO in dataset : 6647


In [ ]:
split_batch_dict['batch_1']['train']

ret  mkt_ret  reversal_1d  mom_scaled_5d  mom_5d  \
date       PERMNO                                                        
2000-10-16 83213  -0.0034   0.0003       0.1364         0.1364  0.2273   
2000-10-17 76087  -0.0065  -0.0179      -0.2317        -0.2927 -0.1463   
           83213  -0.0034  -0.0179      -0.0122        -0.1341 -0.0976   
           83215  -0.0333  -0.0179      -0.4512         0.0244 -0.2195   
           86845  -0.0149  -0.0179       0.2927         0.2439  0.2317   
...                   ...      ...          ...            ...     ...   
2015-12-30 79233  -0.0021  -0.0072       0.0653         0.4032  0.0486   
           81655  -0.0025  -0.0072      -0.2086         0.0770 -0.2176   
           44601  -0.0039  -0.0072      -0.3793        -0.1410 -0.2288   
           35554  -0.0110  -0.0072       0.1342         0.2131  0.0333   
           87034  -0.0103  -0.0072       0.0590         0.4396  0.1784   

                   vol_5d  mom_scaled_21d  mom_21d  vol_21d  mom_scaled_63d  \
date       PERMNO                                                             
2000-10-16 83213   0.0455         -0.1364   0.0455  -0.0455         -0.3182   
2000-10-17 76087  -0.2683         -0.2317  -0.0122  -0.3537          0.1098   
           83213  -0.0366         -0.2073  -0.1098  -0.1463         -0.4512   
           83215   0.3659         -0.0122  -0.1707   0.2317         -0.2439   
           86845   0.3902          0.2805   0.4024   0.3171          0.2927   
...                   ...             ...      ...      ...             ...   
2015-12-30 79233  -0.3856          0.4167   0.3712  -0.3802          0.4365   
           81655  -0.4221          0.4644   0.4671   0.1455          0.0266   
           44601  -0.2212         -0.1599  -0.0928  -0.2527          0.3554   
           35554  -0.1986          0.0455   0.0410  -0.0023         -0.0910   
           87034  -0.3473         -0.3806  -0.2613  -0.1802          0.3099   

                   mom_63d  vol_63d  mom_scaled_126d  mom_126d  vol_126d  \
date       PERMNO                                                          
2000-10-16 83213   -0.2273  -0.2273          -0.3182   -0.3182    0.0455   
2000-10-17 76087    0.0976  -0.4024           0.2317    0.0610   -0.4512   
           83213   -0.3659  -0.1951          -0.4390   -0.4146   -0.1951   
           83215   -0.3049   0.1098          -0.2439   -0.2561    0.0976   
           86845    0.3293   0.1707           0.4390    0.3902    0.0610   
...                    ...      ...              ...       ...       ...   
2015-12-30 79233    0.2883  -0.3797           0.4793    0.4387   -0.3869   
           81655    0.0739  -0.0590           0.1378    0.1342   -0.1194   
           44601    0.2892  -0.2383           0.1878    0.1748   -0.1982   
           35554   -0.0856  -0.1018           0.0721    0.0716   -0.1284   
           87034    0.2743  -0.2126           0.3883    0.4027   -0.1806   

                   mom_scaled_252d  mom_252d  vol_252d  target  
date       PERMNO                                               
2000-10-16 83213           -0.3182   -0.3182    0.2273 -0.0034  
2000-10-17 76087            0.3902    0.1829   -0.4512  0.0000  
           83213           -0.4512   -0.3902   -0.1220 -0.0345  
           83215           -0.3049   -0.3415    0.0244 -0.0287  
           86845            0.3780    0.4390    0.0854 -0.0128  
...                            ...       ...       ...     ...  
2015-12-30 79233            0.1973    0.1743   -0.3392  0.0018  
           81655            0.4068    0.3982   -0.1329 -0.0101  
           44601            0.4054    0.3739   -0.2446 -0.0107  
           35554            0.0302    0.0311   -0.1775 -0.0119  
           87034            0.4802    0.4333   -0.3000 -0.0131  

[1005170 rows x 19 columns]